In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO


# 1. LOAD MODELS 
vehicle_model = YOLO("yolov8n.pt")  

# Mô hình tìm biển số (File cũ của bạn)
plate_model = YOLO(r"C:\Users\Admin\Desktop\dpl\runs\detect\train5\weights\best.pt")

# Mô hình đọc chữ (File bạn vừa tải từ Kaggle về, nhớ đổi tên cho dễ nhớ)
char_model = YOLO(r"C:\Users\Admin\Desktop\dpl\kaggle\working\runs\detect\Plate_OCR\yolov8s_char\weights\best_char.pt") 

# Danh sách 36 ký tự (LƯU Ý: Thứ tự này phải khớp với file data.yaml trên Roboflow của bạn)
char_classes = [
    '0', '1', '2', '3', '4', '5', '6', '7', '8', '9',
    'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 
    'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 
    'U', 'V', 'W', 'X', 'Y', 'Z'
]


# 2. HELPER FUNCTION: XẾP CHỮ BẰNG YOLO

def process_characters(char_results):
    """
    Nhận kết quả từ YOLO char_model, phân loại biển 1 dòng/2 dòng,
    và sắp xếp ký tự từ trái qua phải, trên xuống dưới.
    """
    boxes_data = []
    
    for r in char_results:
        for box in r.boxes:
            conf = float(box.conf[0])
            if conf < 0.3:  # Bỏ qua các nhiễu rác (độ tin cậy thấp)
                continue
                
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            cls_id = int(box.cls[0])
            char = char_classes[cls_id]
            
            # Tính tâm tọa độ và chiều cao của chữ
            cx = (x1 + x2) // 2
            cy = (y1 + y2) // 2
            h = y2 - y1 
            
            boxes_data.append({"char": char, "cx": cx, "cy": cy, "h": h})
            
    if not boxes_data:
        return ""

    # Sắp xếp tạm theo chiều dọc (từ trên xuống)
    boxes_data.sort(key=lambda b: b["cy"])
    
    avg_h = sum(b["h"] for b in boxes_data) / len(boxes_data)
    line_1, line_2 = [], []
    ref_cy = boxes_data[0]["cy"]  # Tâm Y của chữ cao nhất
    
    # Chia thành 2 dòng
    for b in boxes_data:
        # Nếu tâm Y tụt xuống hơn 60% chiều cao chữ -> thuộc dòng 2
        if (b["cy"] - ref_cy) > (avg_h * 0.6):
            line_2.append(b)
        else:
            line_1.append(b)
            
    # Sắp xếp từ trái qua phải cho từng dòng
    line_1.sort(key=lambda b: b["cx"])
    line_2.sort(key=lambda b: b["cx"])
    
    # Ghép chuỗi
    str_line_1 = "".join([b["char"] for b in line_1])
    str_line_2 = "".join([b["char"] for b in line_2])
    
    if len(line_2) > 0:
        return f"{str_line_1}-{str_line_2}"  # VD: 29A1-12345
    else:
        return str_line_1                    # VD: 30A12345

# 3. MAIN PIPELINE

# ĐƯỜNG DẪN ẢNH TEST
image_path = r"C:\Users\Admin\Desktop\dpl\Screenshot 2026-03-10 115447.png"
# C:\Users\Admin\Desktop\dpl\Screenshot 2026-03-10 115447.png
#C:\Users\Admin\Desktop\dpl\Dieu_0046.png
img = cv2.imread(image_path)

# Detect Vehicle
vehicle_results = vehicle_model(img)

plates_found = []

for r in vehicle_results:
    for box in r.boxes:
        cls = int(box.cls[0])
        
        # Lọc xe (Car, Motorbike, Bus, Truck)
        if cls in [2, 3, 5, 7]:  
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            vehicle = img[y1:y2, x1:x2]

            # Bỏ qua nếu khung xe bị lỗi (quá nhỏ/âm)
            if vehicle.size == 0:
                continue

            # Detect Plate trong Vehicle
            plate_results = plate_model(vehicle)

            for pr in plate_results:
                for pbox in pr.boxes:
                    px1, py1, px2, py2 = map(int, pbox.xyxy[0])
                    plate = vehicle[py1:py2, px1:px2]

                    if plate.size == 0:
                        continue

                    
                    # Phóng to biển số 2 lần giúp YOLO nhìn chữ rõ hơn
                    plate_resized = cv2.resize(plate, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)
                    
                    # Cho YOLO đọc ký tự
                    char_results = char_model(plate_resized)
                    
                    # Gọi hàm xếp chữ
                    text = process_characters(char_results)

                    if len(text) >= 4:  # Bỏ qua nếu đọc ra quá ít chữ (biển lỗi/nhiễu)
                        plates_found.append(text)

                    # Vẽ khung biển số và kết quả lên ảnh gốc
                    cv2.rectangle(img, (x1+px1, y1+py1), (x1+px2, y1+py2), (0, 255, 0), 2)
                    cv2.putText(img, text, (x1+px1, y1+py1-10),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)


# OUTPUT

print("Detected plates:", plates_found)

cv2.imshow("Result", img)
cv2.waitKey(0)
cv2.destroyAllWindows()


0: 448x640 3 persons, 1 car, 34.8ms
Speed: 6.7ms preprocess, 34.8ms inference, 120.2ms postprocess per image at shape (1, 3, 448, 640)

0: 384x640 1 License_Plate, 34.5ms
Speed: 0.0ms preprocess, 34.5ms inference, 0.0ms postprocess per image at shape (1, 3, 384, 640)

0: 256x320 2 1s, 2 3s, 1 5, 2 6s, 1 G, 1 W, 37.3ms
Speed: 1.0ms preprocess, 37.3ms inference, 0.0ms postprocess per image at shape (1, 3, 256, 320)
Detected plates: ['51WG-16633']
